## Figures 3-6 (skeleton-based BISINDO benchmark)
Run the setup cell first, then each figure cell. Adjust `DATA`, `KEYPOINTS_CSV` and `O` in the setup cell if needed.

In [ ]:
import os
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# === Run Jupyter from the repository root, or edit these paths ===
DATA = "results/A_outputs/"   # archived result CSVs (included in this repository)
KEYPOINTS_CSV = "data/dataset_keypoints.csv"  # NOT included (see data/README.md); needed for Fig. 3 and 4 only
O    = "figures/"   # output folder for the figures
os.makedirs(O, exist_ok=True)
D = DATA
# files needed in DATA:
#  dataset_keypoints.csv, A_frame_cv_summary.csv, A_delta_cv_summary.csv, A_grouped_cv_summary.csv,
#  Aout_video_level_classical_grouped.csv, A_dl_cv_folds.csv, A_cm_7030_frame285_RF.csv

INK,SUB,SURF="#0b0b0b","#52514e","#fcfcfb"; BLUE="#2a78d6"; ORG="#eb6834"
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":9,"text.color":INK,"axes.edgecolor":"#c9c8c4"})

# Bilingual class labels (same order as in the CSV files)
LABELS = {
    "Halo": "Halo (Hello)",
    "Apa Kabar": "Apa Kabar (How Are You?)",
    "Maaf": "Maaf (Sorry)",
    "Terima kasih": "Terima Kasih (Thank You)",
    "Sama-sama": "Sama-sama (You're Welcome)",
    "Tolong": "Tolong (Help)",
}
def lab(k, wrap=False):
    s = LABELS[k]
    return s.replace(" (", "\n(") if wrap else s


### Fig. 3 & 4 — missing keypoints

In [ ]:
d=pd.read_csv(KEYPOINTS_CSV); X=d[[f"f{i}" for i in range(285)]].to_numpy(float).reshape(len(d),95,3)
m=((X[:,:,0]==0)&(X[:,:,1]==0)); tot=m.sum(1); assert len(d)==6300
seg=[("Pose (33 keypoints)",slice(0,33),33),("Hand slot 1 (21)",slice(33,54),21),("Hand slot 2 (21)",slice(54,75),21),("Face (20)",slice(75,95),20)]
cmap=LinearSegmentedColormap.from_list("b",["#f4f8fd","#2a78d6","#0b2f5e"])
# ---- Figure 3
fig=plt.figure(figsize=(11.5,7.2),dpi=300,facecolor=SURF)
gs=fig.add_gridspec(4,2,width_ratios=[3.4,1],hspace=.55,wspace=.18,left=.14,right=.97,top=.9,bottom=.25)
cls=[lab(k,wrap=True) for k in ["Apa Kabar","Halo","Maaf","Sama-sama","Terima kasih","Tolong"]]
for r,(name,sl,n) in enumerate(seg):
    c=m[:,sl].sum(1); frac=c/n
    ax=fig.add_subplot(gs[r,0]); ax.set_facecolor(SURF)
    im=ax.imshow(frac[None,:],aspect="auto",cmap=cmap,vmin=0,vmax=1,interpolation="nearest")
    ax.set_yticks([]); ax.set_ylabel(name,rotation=0,ha="right",va="center",fontsize=8.5)
    for b in range(1,6): ax.axvline(b*1050-.5,color="white",lw=1.2)
    if r==3: ax.set_xticks([i*1050+525 for i in range(6)]); ax.set_xticklabels(cls,fontsize=8); ax.set_xlabel("Frame index (file order, grouped by class; 1,050 frames per class)",labelpad=4)
    else: ax.set_xticks([])
    for s in ax.spines.values(): s.set_visible(False)
    ax2=fig.add_subplot(gs[r,1]); ax2.set_facecolor(SURF)
    vals=pd.Series(c).value_counts().sort_index(); xs=np.arange(len(vals))
    ax2.bar(xs,vals.values,color=BLUE,width=.55)
    ax2.set_xticks(xs); ax2.set_xticklabels([str(k) for k in vals.index],fontsize=8)
    for x_,v in zip(xs,vals.values): ax2.annotate(f"{v:,}",(x_,v),xytext=(0,2),textcoords="offset points",ha="center",fontsize=7.5,color=SUB)
    ax2.set_ylim(0,6300*1.18); ax2.set_yticks([0,3000,6000]); ax2.grid(axis="y",color="#e6e5e1",lw=.5); ax2.set_axisbelow(True)
    for s_ in("top","right"): ax2.spines[s_].set_visible(False)
    if r==3: ax2.set_xlabel("Missing keypoints in segment")
    if r==0: ax2.set_title("(b) Frames by number missing",fontsize=9,loc="left")
fig.text(.14,.945,"(a) Fraction of segment keypoints missing, per frame",fontsize=9.5,ha="left")
cax=fig.add_axes([.14,.09,.5,.018]); cb=fig.colorbar(im,cax=cax,orientation="horizontal"); cb.set_label("Fraction of the segment's keypoints missing (0 = all detected, 1 = none detected)",fontsize=8); cb.outline.set_visible(False)
fig.savefig(O+"Fig3_missing_keypoints_A.png",facecolor=SURF); fig.savefig(O+"Fig3_missing_keypoints_A.pdf",facecolor=SURF); plt.show()


In [ ]:
# ---- Figure 4
vc=pd.Series(tot).value_counts().sort_index()
fig,ax=plt.subplots(figsize=(7.2,3.9),dpi=300,facecolor=SURF); ax.set_facecolor(SURF)
cols=[BLUE if k<=30 else ORG for k in vc.index]
ax.bar(vc.index,vc.values,color=cols,width=.9)
ax.set_yscale("log"); ax.set_ylim(1,1e4); ax.set_xlim(-3,79)
for k,v in vc.items(): ax.annotate(f"{v:,}",(k,v),xytext=(0,2),textcoords="offset points",ha="center",fontsize=7.5,color=SUB)
ax.axvline(30.5,color=INK,ls="--",lw=1.1); ax.annotate("threshold: keep ≤ 30 missing",(30.5,5000),xytext=(3,0),textcoords="offset points",fontsize=8,va="center")
ax.set_xlabel("Missing keypoints per frame (out of 95)"); ax.set_ylabel("Number of frames (log scale)")
ax.grid(axis="y",color="#e6e5e1",lw=.5); ax.set_axisbelow(True)
for s_ in("top","right"): ax.spines[s_].set_visible(False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=BLUE,label="Kept: 5,379 frames (85.38%)"),Patch(color=ORG,label="Removed: 921 frames (14.62%)")],frameon=False,fontsize=8.5,loc="upper right",bbox_to_anchor=(1,.82))
ax.text(.985,.42,"n = 6,300 frames\nmean 20.54, SD 11.82\nmin 0, max 75\nquartiles 25/50/75: 21 / 21 / 21",transform=ax.transAxes,ha="right",va="top",fontsize=8.5,color=SUB)
fig.tight_layout(); fig.savefig(O+"Fig4_threshold_A.png",facecolor=SURF); fig.savefig(O+"Fig4_threshold_A.pdf",facecolor=SURF); plt.show()
print("ok")

### Fig. 5 — model comparison

In [ ]:
fr=pd.read_csv(D+"A_frame_cv_summary.csv"); de=pd.read_csv(D+"A_delta_cv_summary.csv"); gr=pd.read_csv(D+"A_grouped_cv_summary.csv")
vl=pd.read_csv(D+"Aout_video_level_classical_grouped.csv"); dl=pd.read_csv(D+"A_dl_cv_folds.csv")
rand=pd.concat([fr,de]); 
INK,SUB,SURF="#0b0b0b","#52514e","#fcfcfb"
COL={"frame":"#2a78d6","delta":"#eb6834","dl":"#1baf7a"}
MK={"frame":"o","delta":"s","dl":"D"}
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":10,"axes.edgecolor":"#c9c8c4","text.color":INK})
models=["RF","MLP","SVM","KNN"]
def panel(ax,title,rows,ylim,labels):
    # rows: list of (label,mean,sd,kind)
    ax.set_facecolor(SURF); ax.set_title(title,loc="left",fontsize=10.5,color=INK)
    for i,(lab,m,s,k) in enumerate(rows):
        ax.errorbar(i,m,yerr=s,fmt=MK[k],color=COL[k],ms=8,mec=SURF,mew=1.5,capsize=3,lw=1.6,zorder=3)
        ax.annotate(f"{m:.2f}",(i,m+s),xytext=(0,5),textcoords="offset points",ha="center",fontsize=8,color=SUB)
    ax.set_xticks(range(len(rows))); ax.set_xticklabels([r[0] for r in rows],rotation=35,ha="right",fontsize=8.5)
    ax.set_ylim(*ylim); ax.set_ylabel("Accuracy (%)")
    ax.grid(axis="y",color="#e6e5e1",lw=.6); ax.set_axisbelow(True)
    for s_ in("top","right"): ax.spines[s_].set_visible(False)
def rows_from(df):
    r=[]
    for f,k in(("frame285","frame"),("delta855","delta")):
        for m in models:
            x=df[(df.features==f)&(df.model==m)].iloc[0]; r.append((f"{m}\n{'frame' if k=='frame' else 'delta'}",x["mean"],x["sd_ddof0"],k))
    return r
fig,ax=plt.subplots(1,3,figsize=(17,5.2),dpi=300,facecolor=SURF,gridspec_kw={"width_ratios":[8,8,8.5]})
panel(ax[0],"(a) Frame rows, random stratified 5-fold\n(optimistic: same-video frames can cross folds)",rows_from(rand),(84,101),None)
panel(ax[1],"(b) Frame rows, video-grouped 5-fold",rows_from(gr),(80,101),None)
# panel c: video level
r=[]
for f,k in(("frame285","frame"),("delta855","delta")):
    for m in models:
        x=vl[(vl.features==f)&(vl.model==m)].iloc[0]; r.append((f"{m}\n{'frame' if k=='frame' else 'delta'}",x.fold_mean,x.fold_sd_ddof0,k))
acc=dl.assign(a=dl.accuracy*(100 if dl.accuracy.max()<=1 else 1))
for m,n in(("1D-CNN","1D-CNN"),("LSTM","LSTM")):
    d=acc[acc.model==m] if "model" in acc else None
    r.append((n,d.a.mean(),d.a.std(ddof=0),"dl"))
panel(ax[2],"(c) Video level (210 videos), 5-fold\nclassical = majority vote over grouped folds",r,(90,102),None)
for a in ax: a.set_facecolor(SURF)
from matplotlib.lines import Line2D
h=[Line2D([],[],marker=MK[k],color=COL[k],ls="",ms=8,label=l) for k,l in(("frame","Frame features (285-d)"),("delta","Frame + delta features (855-d)"),("dl","Deep-learning baseline (video sequence)"))]
fig.legend(handles=h,loc="lower center",ncol=3,frameon=False,fontsize=9.5,bbox_to_anchor=(.5,-.01))
fig.text(.01,.005,"",fontsize=8)
fig.tight_layout(rect=(0,.06,1,1))
fig.savefig(O+"Fig5_model_comparison_A.png",facecolor=SURF); fig.savefig(O+"Fig5_model_comparison_A.pdf",facecolor=SURF); plt.show()

### Fig. 6 — confusion matrix RF 70:30

In [ ]:
# Fig 6 confusion matrix
cm=pd.read_csv(D+"A_cm_7030_frame285_RF.csv",index_col=0)
fig,ax=plt.subplots(figsize=(8.6,6.2),dpi=300,facecolor=SURF)
v=cm.values; n=v.sum(1,keepdims=True)
im=ax.imshow(v/n,cmap="Blues",vmin=0,vmax=1)
for i in range(6):
    for j in range(6):
        ax.text(j,i,str(v[i,j]),ha="center",va="center",fontsize=10,color="white" if v[i,j]/n[i,0]>.55 else INK)
ax.set_xticks(range(6)); ax.set_yticks(range(6)); ax.set_xticklabels([lab(k) for k in cm.columns],rotation=35,ha="right"); ax.set_yticklabels([lab(k) for k in cm.index])
ax.set_xlabel("Predicted class"); ax.set_ylabel("True class",labelpad=8)
cb=fig.colorbar(im,ax=ax,fraction=.046); cb.set_label("Row-normalised share")
fig.tight_layout(); fig.savefig(O+"Fig6_confusion_RF_7030_A.png",facecolor=SURF); fig.savefig(O+"Fig6_confusion_RF_7030_A.pdf",facecolor=SURF); plt.show()
print("ok",v.trace(),v.sum())